# SP Correlation — full analysis

League 41657 (Il Nuovo Vesuvio) Yahoo scoring applied. Train on 2024 + 2025 May15+ starts (≥5 starts/season, ≥4 IP, ≥3 prior starts entering each row). Project 2026 going forward.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
from src.analysis import (
    load_dataset, univariate_correlations, single_metric_models, best_pair,
    fit_model, evaluate, stability, regression_candidates, FEATURE_COLS,
)

In [ ]:
df_24 = load_dataset([2024])
df_25 = load_dataset([2025])
df_26 = load_dataset([2026])
print(f'2024: {len(df_24)} rows, {df_24.pitcher_id.nunique()} SPs')
print(f'2025: {len(df_25)} rows, {df_25.pitcher_id.nunique()} SPs')
print(f'2026: {len(df_26)} rows, {df_26.pitcher_id.nunique()} SPs')

## 1. Univariate correlations (combined 2024+2025)

In [ ]:
combined = pd.concat([df_24, df_25], ignore_index=True)
corr = univariate_correlations(combined)
corr.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
c = corr.set_index('feature')['pearson_r'].sort_values()
ax.barh(c.index, c.values)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Pearson r vs FP per start')
ax.set_title('Univariate correlations — 2024+2025 combined')
plt.tight_layout()

## 2. Single-metric out-of-sample (train 2024, test 2025)

In [ ]:
single = single_metric_models(df_24, df_25)
single.round(3)

## 3. Best 2-metric pairs

In [ ]:
pair = best_pair(df_24, df_25, top_k=8)
pair.head(15).round(3)

## 4. Full multivariate Ridge

In [ ]:
feats = [f for f in FEATURE_COLS if f in df_24.columns]
model, _ = fit_model(df_24, feats)
print('Train (2024):', evaluate(model, feats, df_24))
print('Test  (2025):', evaluate(model, feats, df_25))
if len(df_26):
    print('Test  (2026):', evaluate(model, feats, df_26))

coef = pd.Series(model.coef_, index=feats).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(coef.index, coef.values)
ax.axvline(0, color='black', lw=0.5)
ax.set_title('Ridge coefficients (standardize features for direct comparison)')
plt.tight_layout()

## 5. Non-linearity check — bucket each metric into quintiles

In [ ]:
top_metrics = corr.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, f in zip(axes.flatten(), top_metrics):
    sub = combined[[f, 'fp']].dropna()
    sub['q'] = pd.qcut(sub[f], 5, labels=False, duplicates='drop')
    g = sub.groupby('q')['fp'].agg(['mean', 'count'])
    g['mean'].plot(kind='bar', ax=ax)
    ax.set_title(f)
    ax.set_xlabel('quintile (low→high)')
    ax.set_ylabel('mean FP')
plt.tight_layout()

## 6. Stability (year-over-year)

In [ ]:
stab = stability({2024: df_24, 2025: df_25})
stab.round(3)

## 7. Regression candidates in 2026 (buy / sell)

In [ ]:
if len(df_26):
    cands = regression_candidates(model, feats, df_26)
    print('BUY candidates (actual < projected):')
    display(cands.head(15).round(2))
    print('\nSELL candidates (actual > projected):')
    display(cands.tail(15).iloc[::-1].round(2))